# LGBM Mortality Prediction: Out-of-Time Validation

**Objective**: Evaluate the model's ability to forecast future mortality rates.

- **Train Set**: Observation Years **2009 - 2016**
- **Skip**: Observation Year **2017** (Gap year to simulate reporting lag)
- **Test Set**: Observation Years **2018 - 2019** (Future unseen data)

This simulates a real-world actuarial scenario where historic data is used to price or reserve for future years.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

In [2]:
# Load cleaned data
DATA_PATH = '../data/ilec_cleaned.parquet'
df = pd.read_parquet(DATA_PATH)

# Check year range
print(f"Data Year Range: {df['Observation_Year'].min()} - {df['Observation_Year'].max()}")

## 2. Temporal Split (Train/Test)

In [3]:
# Sort by year to ensure temporal order (good practice, though we split by mask)
df = df.sort_values('Observation_Year')

# Define Split Logic
train_mask = (df['Observation_Year'] >= 2009) & (df['Observation_Year'] <= 2016)
test_mask = (df['Observation_Year'] >= 2018) & (df['Observation_Year'] <= 2019)

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

print(f"Train Years: {df_train['Observation_Year'].unique()}")
print(f"Train Samples: {len(df_train):,}")
print("-" * 30)
print(f"Test Years: {df_test['Observation_Year'].unique()}")
print(f"Test Samples: {len(df_test):,}")

## 3. Feature Engineering & Offset Calculation

In [4]:
NUMERICAL_FEATURES = ['Attained_Age', 'Issue_Age', 'Duration']
CATEGORICAL_FEATURES = [
    'Sex', 'Smoker_Status', 'Insurance_Plan', 'Face_Amount_Band',
    'Preferred_Class', 'SOA_Post_Lvl_Ind', 'SOA_Antp_Lvl_TP', 'SOA_Guar_Lvl_TP'
]
TARGET = 'Death_Count'
EXPOSURE = 'Policies_Exposed'

# Function to prepare datasets
def prepare_dataset(data):
    X = data[NUMERICAL_FEATURES + CATEGORICAL_FEATURES].copy()
    y = data[TARGET].values
    exposure = data[EXPOSURE].values
    # Offset = log(Exposure)
    offset = np.log(exposure + 1e-10)
    
    # Convert categoricals
    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].astype('category')
        
    return X, y, offset, exposure

X_train, y_train, offset_train, exposure_train = prepare_dataset(df_train)
X_test, y_test, offset_test, exposure_test = prepare_dataset(df_test)

print("Datasets prepared.")

## 4. Train LightGBM Model (Offset Poisson)

In [5]:
# Create LGBM Datasets
lgb_train = lgb.Dataset(
    X_train, 
    label=y_train,
    init_score=offset_train,
    categorical_feature=CATEGORICAL_FEATURES
)

# We use Test set as valid set to monitor overfitting on future data
lgb_test = lgb.Dataset(
    X_test, 
    label=y_test,
    init_score=offset_test,
    categorical_feature=CATEGORICAL_FEATURES,
    reference=lgb_train
)

params = {
    'objective': 'poisson',
    'metric': 'poisson',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_data_in_leaf': 100,
    'verbose': -1,
    'seed': 42
}

print("Training model...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_test],
    valid_names=['train', 'future_test'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

## 5. Evaluate Future Predictions (2018-2019)

In [6]:
# Generate Predictions
# predict returns Rate (per unit exposure)
pred_rate_test = model.predict(X_test)
pred_deaths_test = pred_rate_test * exposure_test

# Add predictions back to df_test for aggregation
df_test['Predicted_Deaths'] = pred_deaths_test
df_test['Predicted_Rate'] = pred_rate_test

# Overall A/E
ae_ratio_overall = df_test['Death_Count'].sum() / df_test['Predicted_Deaths'].sum()
print(f"Overall A/E Ratio (2018-2019): {ae_ratio_overall:.4f}")

# Breakdown by Year
yearly_ae = df_test.groupby('Observation_Year').agg({
    'Death_Count': 'sum',
    'Predicted_Deaths': 'sum',
    'Policies_Exposed': 'sum'
}).reset_index()

yearly_ae['AE_Ratio'] = yearly_ae['Death_Count'] / yearly_ae['Predicted_Deaths']
print("\n--- A/E Ratio by Future Year ---")
print(yearly_ae)

## 6. Visualization: Actual vs Expected by Year

In [7]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

years = yearly_ae['Observation_Year'].astype(str)

# Top: Counts
ax1.bar(years, yearly_ae['Death_Count'], alpha=0.6, label='Actual Deaths', color='navy')
ax1.bar(years, yearly_ae['Predicted_Deaths'], alpha=0.4, label='Predicted Deaths', color='orange', width=0.5)
ax1.set_title('Actual vs Predicted Deaths (2018-2019)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bottom: A/E Ratio
ax2.plot(years, yearly_ae['AE_Ratio'], marker='o', color='red', linewidth=2)
ax2.axhline(1.0, linestyle='--', color='gray')
ax2.set_title('A/E Ratio (Target = 1.0)')
ax2.set_xlabel('Observation Year')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0.9, 1.1)  # Zoom in to see deviation

plt.tight_layout()
plt.show()